# Dataset Distribution & Calorimeter Ring Energy Profiles

Organized, memory-efficient streaming pipeline to analyze egamma calorimeter data across $E_T \times \eta$ regions:
1. **Region Selection**: Set `TARGET_REGIONS` to a list of `(et, eta)` pairs (or `None` for all regions) for per-region inspection, and `PROCESS_ALL_REGIONS = True` only when you want to run the global all-regions tasks.
2. **Task 1 — Dataset Variable Distributions**: Unfilled step histograms (`histtype='step'`) per selected region and optional global all-regions summary & event yields.
3. **Task 2 — 100-Ring Energy Profiles**: Per-region and optional global mean energy profiles using `ProfileMeanEnergyPlotter`.
4. **Task 3 — 50-Ring Energy Profiles**: Per-region and optional global mean energy profiles using `ProfileMeanEnergyPlotter`.

In [1]:
import gc
import os
import sys
from pathlib import Path
from types import SimpleNamespace

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh

mh.style.use("ATLAS")

from src.core.Datasets.EgammaNpzDataset import EgammaNpzDataset
from src.core.Plotting.Localization import PlotTextKey, PlotTranslator
from src.core.Plotting.ProfilePlotter import (
    ProfileMeanEnergyPlotter,
    StreamingRingProfileAccumulator,
)
from src.core.Plotting.Context import RegionPlotContext
from src.utils import get_et_eta

print(f"Project root: {PROJECT_ROOT}")

Project root: /eos/home-i04/p/pmourafr/egamma_master


### 1. Configuration, Region Selection & Styling
Configure dataset paths, target `(et, eta)` region list, global execution flag (`PROCESS_ALL_REGIONS`), target observable, and histogram styling.

In [ ]:
CONFIG_PATH_100 = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_100Rings.yaml"
CONFIG_PATH_50 = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_50Rings.yaml"

with open(CONFIG_PATH_100, "r") as f:
    config_dict_100 = yaml.safe_load(f)
with open(CONFIG_PATH_50, "r") as f:
    config_dict_50 = yaml.safe_load(f)

DATA_PATH = config_dict_100["dataset"]["parameters"]["drive_path"]
if not os.path.exists(DATA_PATH):
    local_data_path = PROJECT_ROOT / "data/consolidated"
    if local_data_path.exists():
        DATA_PATH = str(local_data_path)

print(f"Dataset path: {DATA_PATH}")

# Region selection for per-region cells: list of (et, eta) pairs, or None to process all regions
TARGET_REGIONS = [(0, 0)]



LANGUAGE = PlotTranslator.validate_language("pt")
print(f"Plot language: {LANGUAGE}")

TARGET_VARIABLE = "avgmu"

HIST_BINS = 50
NORMED_DENSITY = True
HIST_TYPE = "step"
HIST_LINEWIDTH = 2.0
LOG_SCALE = True

OUTPUT_DIR = PROJECT_ROOT / "Plots/dataset_distributions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 2. Dataset Instantiation & Region Resolution
Instantiate `dataset_100` and `dataset_50`, inspect available columns, and resolve the selected `(et, eta)` region file indices.

In [ ]:
params_100 = config_dict_100["dataset"]["parameters"]
params_50 = config_dict_50["dataset"]["parameters"]

dataset_100 = EgammaNpzDataset(
    drive_path=DATA_PATH,
    startswith=params_100.get("startswith", "consolidated.et"),
    endswith=params_100.get("endswith", ".npz"),
    percentage=params_100.get("percentage", 1.0),
)

dataset_50 = EgammaNpzDataset(
    drive_path=DATA_PATH,
    startswith=params_50.get("startswith", "consolidated.et"),
    endswith=params_50.get("endswith", ".npz"),
    percentage=params_50.get("percentage", 0.5),
)

feature_names = dataset_100.feature_names
dataset_100._set_ring_column_indices(feature_names)
dataset_50._set_ring_column_indices(feature_names)

if TARGET_VARIABLE not in feature_names:
    raise ValueError(f"Variable '{TARGET_VARIABLE}' not found in dataset! Available: {list(feature_names)}")

target_var_idx = int(np.where(feature_names == TARGET_VARIABLE)[0][0])

region_index_map = {
    tuple(int(v) for v in get_et_eta(fp)): idx
    for idx, fp in enumerate(dataset_100.file_paths)
}

if TARGET_REGIONS is None:
    selected_indices = list(range(len(dataset_100)))
else:
    missing = [pair for pair in TARGET_REGIONS if tuple(pair) not in region_index_map]
    if missing:
        raise ValueError(f"Requested (et, eta) regions not found: {missing}. Available: {sorted(region_index_map)}")
    selected_indices = [region_index_map[tuple(pair)] for pair in TARGET_REGIONS]

profile_plotter = ProfileMeanEnergyPlotter(language=LANGUAGE)
sig_label = PlotTranslator.translate(PlotTextKey.PROFILE_SIGNAL_LABEL, language=LANGUAGE)
bg_label = PlotTranslator.translate(PlotTextKey.PROFILE_BACKGROUND_LABEL, language=LANGUAGE)

print(f"Discovered region files: {len(dataset_100)}")
print(f"Selected region indices ({len(selected_indices)}): {selected_indices}")
print(f"Active rings in 100-ring model: {dataset_100.percentage_dim}")
print(f"Active rings in 50-ring model:  {dataset_50.percentage_dim}")
print(f"Target variable '{TARGET_VARIABLE}' column index: {target_var_idx}")

### 3. Task 1: Dataset Variable Distribution Plots (Per-Region)
Stream each selected `(et, eta)` region file, mask `TARGET_VARIABLE`, plot signal vs. background step histograms, and free memory.

In [ ]:
def plot_variable_distribution(
    var_signal: np.ndarray,
    var_bg: np.ndarray,
    title: str,
    output_pdf: Path,
    figsize: tuple[float, float] = (8, 5),
) -> None:
    """Plot unfilled step histogram comparing signal and background distributions."""
    mh.style.use("ATLAS")
    q_low = min(float(np.percentile(var_signal, 0.1)), float(np.percentile(var_bg, 0.1)))
    q_high = max(float(np.percentile(var_signal, 99.9)), float(np.percentile(var_bg, 99.9)))
    val_range = (q_low, q_high)

    fig, ax = plt.subplots(figsize=figsize)
    ax.hist(
        var_signal, bins=HIST_BINS, range=val_range, density=NORMED_DENSITY,
        histtype=HIST_TYPE, linewidth=HIST_LINEWIDTH, color="navy",
        label=f"{sig_label} (N={len(var_signal):,})",
    )
    ax.hist(
        var_bg, bins=HIST_BINS, range=val_range, density=NORMED_DENSITY,
        histtype=HIST_TYPE, linewidth=HIST_LINEWIDTH, color="darkorange",
        label=f"{bg_label} (N={len(var_bg):,})",
    )
    if LOG_SCALE:
        ax.set_yscale("log")
    ax.set_xlabel(TARGET_VARIABLE, fontsize=12)
    y_base_key = PlotTextKey.DATASET_DIST_YLABEL_DENSITY if NORMED_DENSITY else PlotTextKey.DATASET_DIST_YLABEL_COUNTS
    y_label = PlotTranslator.translate(y_base_key, language=LANGUAGE)
    if LOG_SCALE:
        y_label += PlotTranslator.translate(PlotTextKey.DATASET_DIST_LOG_SUFFIX, language=LANGUAGE)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(loc="upper right", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(output_pdf, bbox_inches="tight")
    print(f"  Distribution plot saved to: {output_pdf}")
    plt.show()

var_output_dir = OUTPUT_DIR / TARGET_VARIABLE
var_output_dir.mkdir(parents=True, exist_ok=True)

for idx in selected_indices:
    data, labels, file_path = dataset_100[idx]
    et, eta = get_et_eta(file_path)
    region_key = f"et{int(et)}_eta{int(eta)}"

    signal_mask = labels == 1
    bg_mask = labels == 0
    var_signal = data[signal_mask, target_var_idx]
    var_bg = data[bg_mask, target_var_idx]

    print(f"{'='*70}")
    print(f"Region {region_key} (ET={et}, ETA={eta}) | Signal={len(var_signal):,}, Background={len(var_bg):,}")

    title = PlotTranslator.translate(
        PlotTextKey.DATASET_DIST_REGIONAL_TITLE,
        language=LANGUAGE,
        variable=TARGET_VARIABLE,
        region=region_key,
    )
    plot_variable_distribution(
        var_signal=var_signal,
        var_bg=var_bg,
        title=title,
        output_pdf=var_output_dir / f"{region_key}_{TARGET_VARIABLE}_distribution.pdf",
    )

    del data, labels, var_signal, var_bg
    gc.collect()

### 4. Task 1b: Global Variable Distribution & Regional Event Yields (All Regions)
Executes **only when `PROCESS_ALL_REGIONS = True`**. Streams all region files to build the consolidated `TARGET_VARIABLE` distribution and regional event yield bar chart.

In [ ]:
# Global all-regions flag: set True to run the global cells (re-streams all files); default False
PROCESS_ALL_REGIONS = False

if not PROCESS_ALL_REGIONS:
    print("Skipping global variable distribution plot (PROCESS_ALL_REGIONS=False).")
else:
    global_signal_values = []
    global_bg_values = []
    region_summaries = []

    for idx in range(len(dataset_100)):
        data, labels, file_path = dataset_100[idx]
        et, eta = get_et_eta(file_path)
        region_key = f"et{int(et)}_eta{int(eta)}"

        signal_mask = labels == 1
        bg_mask = labels == 0
        var_signal = data[signal_mask, target_var_idx].copy()
        var_bg = data[bg_mask, target_var_idx].copy()

        global_signal_values.append(var_signal)
        global_bg_values.append(var_bg)
        region_summaries.append({
            "region": region_key,
            "et": int(et),
            "eta": int(eta),
            "n_signal": int(len(var_signal)),
            "n_bg": int(len(var_bg)),
            "total": int(len(var_signal) + len(var_bg)),
        })

        del data, labels
        gc.collect()

    all_signal = np.concatenate(global_signal_values)
    all_bg = np.concatenate(global_bg_values)
    total_sig_events = len(all_signal)
    total_bg_events = len(all_bg)
    total_events = total_sig_events + total_bg_events

    print(f"{'='*70}")
    print("ALL REGIONS CONSOLIDATED MACHINE BEHAVIOR SUMMARY")
    print(f"{'='*70}")
    print(f"Total Regions Processed:        {len(region_summaries)}")
    print(f"Total Events in Dataset:        {total_events:,}")
    print(f"Total Signal Events (Photons):  {total_sig_events:,} ({total_sig_events / total_events * 100:.2f}%)")
    print(f"Total Background Events (Jets): {total_bg_events:,} ({total_bg_events / total_events * 100:.2f}%)")
    print(f"{TARGET_VARIABLE} Global Range:        [{all_signal.min():.3f}, {all_signal.max():.3f}]")

    global_title = PlotTranslator.translate(
        PlotTextKey.DATASET_DIST_GLOBAL_TITLE,
        language=LANGUAGE,
        variable=TARGET_VARIABLE,
        n_regions=len(region_summaries),
    )
    plot_variable_distribution(
        var_signal=all_signal,
        var_bg=all_bg,
        title=global_title,
        output_pdf=var_output_dir / f"global_{TARGET_VARIABLE}_distribution_all_regions.pdf",
        figsize=(9, 5.5),
    )

    df_summary = pd.DataFrame(region_summaries)
    if len(df_summary) > 1:
        mh.style.use("ATLAS")
        fig_bar, ax_bar = plt.subplots(figsize=(14, 5))
        x_indices = np.arange(len(df_summary))
        width = 0.35
        ax_bar.bar(x_indices - width / 2, df_summary["n_signal"], width, label=sig_label, color="navy", alpha=0.8)
        ax_bar.bar(x_indices + width / 2, df_summary["n_bg"], width, label=bg_label, color="darkorange", alpha=0.8)
        ax_bar.set_xticks(x_indices)
        ax_bar.set_xticklabels(df_summary["region"], rotation=45, ha="right", fontsize=10)
        ax_bar.set_xlabel(PlotTranslator.translate(PlotTextKey.DATASET_YIELD_XLABEL, language=LANGUAGE), fontsize=11)
        ax_bar.set_ylabel(PlotTranslator.translate(PlotTextKey.DATASET_YIELD_YLABEL, language=LANGUAGE), fontsize=11)
        ax_bar.set_title(PlotTranslator.translate(PlotTextKey.DATASET_YIELD_TITLE, language=LANGUAGE), fontsize=13)
        ax_bar.legend()
        ax_bar.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

### 5. Task 2: 100-Ring Calorimeter Energy Profiles (Per-Region)
Stream each selected `(et, eta)` region file and plot the 100-ring mean energy profile.

In [ ]:
mock_trainer_100 = SimpleNamespace(full_dataset=dataset_100)
rings_100_cols = dataset_100.ring_column_indices

for idx in selected_indices:
    data, labels, file_path = dataset_100[idx]
    et, eta = get_et_eta(file_path)
    region_key = f"et{int(et)}_eta{int(eta)}"

    rings_100_data = data[:, rings_100_cols]
    context_100 = RegionPlotContext(
        iet=int(et),
        ieta=int(eta),
        output_dir=OUTPUT_DIR,
        data=data,
        target=labels,
        trainer=mock_trainer_100,
        language=LANGUAGE,
    )
    title_100 = PlotTranslator.translate(
        PlotTextKey.PROFILE_REGIONAL_TITLE,
        language=LANGUAGE,
        n_rings=dataset_100.percentage_dim,
        region=region_key,
    )
    pdf_100 = profile_plotter.plot(
        context_100,
        x_rings=rings_100_data,
        plot_name="RingsMeanProfiles_100Rings",
        plot_title=title_100,
        show=True,
    )
    print(f"[100 Rings] {region_key} saved: {pdf_100}")

    del data, labels, rings_100_data
    gc.collect()

### 6. Task 2b: Global 100-Ring Calorimeter Energy Profile (All Regions)
Executes **only when `PROCESS_ALL_REGIONS = True`**. Streams all region files to compute and plot the global 100-ring energy profile.

In [ ]:
# Global all-regions flag: set True to run the global cells (re-streams all files); default False
PROCESS_ALL_REGIONS = False

if not PROCESS_ALL_REGIONS:
    print("Skipping global 100-ring energy profile (PROCESS_ALL_REGIONS=False).")
else:
    acc_100 = StreamingRingProfileAccumulator(
        n_rings=dataset_100.percentage_dim,
        ring_indexes=dataset_100.indexes,
    )
    rings_100_cols = dataset_100.ring_column_indices

    for idx in range(len(dataset_100)):
        data, labels, _ = dataset_100[idx]
        acc_100.update(data[:, rings_100_cols], labels)
        del data, labels
        gc.collect()

    mean_sig_100, std_sig_100, mean_bg_100, std_bg_100 = acc_100.compute_stats()
    global_title_100 = PlotTranslator.translate(
        PlotTextKey.PROFILE_GLOBAL_TITLE,
        language=LANGUAGE,
        n_rings=acc_100.n_rings,
        n_regions=len(dataset_100),
    )
    global_pdf_100 = profile_plotter.plot_from_stats(
        mean_signal=mean_sig_100,
        std_signal=std_sig_100,
        mean_bkg=mean_bg_100,
        std_bkg=std_bg_100,
        output_dir=OUTPUT_DIR,
        filename="global_RingsMeanProfiles_100Rings.pdf",
        ring_indexes=acc_100.ring_indexes,
        plot_title=global_title_100,
        language=LANGUAGE,
        show=True,
    )
    print(f"Saved global 100-ring energy profile to: {global_pdf_100}")

### 7. Task 3: 50-Ring Calorimeter Energy Profiles (Per-Region)
Stream each selected `(et, eta)` region file and plot the 50-ring mean energy profile.

In [ ]:
mock_trainer_50 = SimpleNamespace(full_dataset=dataset_50)
rings_50_cols = dataset_50.ring_column_indices

for idx in selected_indices:
    data, labels, file_path = dataset_100[idx]
    et, eta = get_et_eta(file_path)
    region_key = f"et{int(et)}_eta{int(eta)}"

    rings_50_data = data[:, rings_50_cols]
    context_50 = RegionPlotContext(
        iet=int(et),
        ieta=int(eta),
        output_dir=OUTPUT_DIR,
        data=data,
        target=labels,
        trainer=mock_trainer_50,
        language=LANGUAGE,
    )
    title_50 = PlotTranslator.translate(
        PlotTextKey.PROFILE_REGIONAL_TITLE,
        language=LANGUAGE,
        n_rings=dataset_50.percentage_dim,
        region=region_key,
    )
    pdf_50 = profile_plotter.plot(
        context_50,
        x_rings=rings_50_data,
        plot_name="RingsMeanProfiles_50Rings",
        plot_title=title_50,
        show=True,
    )
    print(f"[50 Rings] {region_key} saved: {pdf_50}")

    del data, labels, rings_50_data
    gc.collect()

### 8. Task 3b: Global 50-Ring Calorimeter Energy Profile (All Regions)
Executes **only when `PROCESS_ALL_REGIONS = True`**. Streams all region files to compute and plot the global 50-ring energy profile.

In [ ]:
# Global all-regions flag: set True to run the global cells (re-streams all files); default False
PROCESS_ALL_REGIONS = False

if not PROCESS_ALL_REGIONS:
    print("Skipping global 50-ring energy profile (PROCESS_ALL_REGIONS=False).")
else:
    acc_50 = StreamingRingProfileAccumulator(
        n_rings=dataset_50.percentage_dim,
        ring_indexes=dataset_50.indexes,
    )
    rings_50_cols = dataset_50.ring_column_indices

    for idx in range(len(dataset_100)):
        data, labels, _ = dataset_100[idx]
        acc_50.update(data[:, rings_50_cols], labels)
        del data, labels
        gc.collect()

    mean_sig_50, std_sig_50, mean_bg_50, std_bg_50 = acc_50.compute_stats()
    global_title_50 = PlotTranslator.translate(
        PlotTextKey.PROFILE_GLOBAL_TITLE,
        language=LANGUAGE,
        n_rings=acc_50.n_rings,
        n_regions=len(dataset_100),
    )
    global_pdf_50 = profile_plotter.plot_from_stats(
        mean_signal=mean_sig_50,
        std_signal=std_sig_50,
        mean_bkg=mean_bg_50,
        std_bkg=std_bg_50,
        output_dir=OUTPUT_DIR,
        filename="global_RingsMeanProfiles_50Rings.pdf",
        ring_indexes=acc_50.ring_indexes,
        plot_title=global_title_50,
        language=LANGUAGE,
        show=True,
    )
    print(f"Saved global 50-ring energy profile to: {global_pdf_50}")